In [ ]:
import os
if "COLAB_" in "".join(os.environ.keys()):
    print("Running in Google Colab environment.")
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    !pip install evaluate sacrebleu trackio
else:
    from dotenv import load_dotenv
    load_dotenv()

Running in Google Colab environment.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset

- Load NLLB model

In [4]:
dataset = load_dataset("madoss/fr-mos-final-data")

In [5]:
dataset = dataset.rename_column("source", "data_source")
dataset = dataset.rename_column("french", "source")
dataset = dataset.rename_column("moore", "target")

In [6]:
src_lang = "fra_Latn"
tgt_lang = "mos_Latn"

def add_language_info(example):
    example["source_lang"] = src_lang
    example["target_lang"] = tgt_lang
    return example

dataset = dataset.map(add_language_info)

In [7]:
model_name = "facebook/nllb-200-distilled-600M"

model = AutoModelForSeq2SeqLM.from_pretrained(
                                            model_name,
                                            device_map='auto',
                                            use_cache=False
                                            )

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [32]:
french = "Tu es mon semblable."
moore = "Foo la mam bogre."
english = "You are my fellow."

In [33]:

tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang=src_lang, tgt_lang=tgt_lang)

In [34]:
simple_example = tokenizer(french, src_lang=src_lang, tgt_lang=tgt_lang, text_target=moore)

In [ ]:
text = tokenizer.decode(simple_example["input_ids"], skip_special_tokens=False)
print(text)
labels = tokenizer.decode(simple_example["labels"], skip_special_tokens=False)
print(labels)

fra_Latn Tu es mon semblable.</s>
mos_Latn Foo la mam bogre.</s>


In [ ]:
inputs = tokenizer(french, src_lang="fra_Latn", return_tensors="pt")
outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("mos_Latn")
)
translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(translation)

Yãmb yaa wa maam.


In [21]:
def tokenize_fn(examples):
    inputs = examples["source"]
    targets = examples["target"]
    src_langs = examples["source_lang"]
    tgt_langs = examples["target_lang"]

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for src, tgt, src_lang, tgt_lang in zip(inputs, targets, src_langs, tgt_langs):
        tokenized = tokenizer(
            src,
            text_target=tgt,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
            max_length=256,
            padding="max_length",
            truncation=True,
        )
        input_ids_list.append(tokenized["input_ids"])
        attention_mask_list.append(tokenized["attention_mask"])
        labels_list.append(tokenized["labels"])
    return {"input_ids": input_ids_list, "attention_mask": attention_mask_list, "labels": labels_list}

In [12]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = [[l] for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    return {
        "bleu": bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"],
        "chrf": chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"],
    }

In [27]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import os

epochs = 1
learning_rate = 2e-5
batch_size = 1

hf_id = "madoss"
output_dir = os.path.join(hf_id, "nllb-200-finetuned-600-FRA-MOS")

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=8,
    eval_accumulation_steps=4,
    gradient_checkpointing=True,

    fp16=True,
    fp16_full_eval=True,

    learning_rate=learning_rate,
    lr_scheduler_type='constant',  # "constant", "linear", "cosine"

    eval_strategy="steps",  # or "epoch"
    eval_steps=100,
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=128,
    report_to="trackio", # "tensorboard", "wandb", "trackio" or "none"
    # push to hub parameters
    push_to_hub=True,
    hub_private_repo=True,
    hub_strategy="end",  # "every_save" or "end"
    hub_model_id=output_dir,
)

In [23]:
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset"
)


Tokenizing dataset:   0%|          | 0/32579 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/2493 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/2575 [00:00<?, ? examples/s]

In [28]:
import torch
torch.cuda.empty_cache()

In [29]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()